# Retail Sales Forecasting Project

This notebook presents a complete data science workflow—from business understanding and data collection to exploratory data analysis (EDA), feature engineering, machine learning model building, and interpretation. The project focuses on forecasting weekly sales for a retail chain using historical sales data, store characteristics, and external factors.

## 1. Problem Definition & Business Understanding

- **Objective:** Forecast weekly sales and understand key drivers (e.g., markdown promotions, holidays, economic factors) affecting sales performance.
- **Key Questions:**
  - What factors (store type, markdowns, holidays, economic indicators) drive sales?
  - How do promotional markdowns impact weekly sales?
  - Can we predict future sales accurately using historical data?
- **Success Metrics:** RMSE, MAE, R², and actionable insights for decision making.

## 2. Data Collection & Integration

We have three datasets:

1. **features.csv**: Contains environmental and economic data (Temperature, Fuel_Price, CPI, Unemployment, MarkDowns, and IsHoliday).
2. **train.csv**: Contains weekly sales data (Store, Dept, Date, Weekly_Sales, IsHoliday).
3. **stores.csv**: Contains store characteristics (Store, Type, Size).

The datasets are merged using common keys (Store, Date) to produce a final dataset for analysis. In this notebook, we assume the merged dataset is available as `output2.csv`.

In [7]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configure visualization style
sns.set(style="whitegrid")

# Load the merged dataset
df = pd.read_excel("output1.xlsx")

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Display basic information
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Unnamed: 0    421570 non-null  int64         
 1   Store         421570 non-null  int64         
 2   Dept          421570 non-null  int64         
 3   Date          421570 non-null  datetime64[ns]
 4   Weekly_Sales  421570 non-null  float64       
 5   IsHoliday_x   421570 non-null  bool          
 6   Temperature   421570 non-null  float64       
 7   Fuel_Price    421570 non-null  float64       
 8   MarkDown1     150681 non-null  float64       
 9   MarkDown2     111248 non-null  float64       
 10  MarkDown3     137091 non-null  float64       
 11  MarkDown4     134967 non-null  float64       
 12  MarkDown5     151432 non-null  float64       
 13  CPI           421570 non-null  float64       
 14  Unemployment  421570 non-null  float64       
 15  IsHoliday_y   421

## 3. Data Cleaning & Preprocessing

- **Check for missing values** and handle them appropriately (e.g., fill missing markdowns with 0 if no discount was applied).
- **Convert data types** (e.g., ensure the Date column is datetime).
- **Remove duplicate columns** (e.g., duplicate IsHoliday if present).

In [ ]:
# Check missing values
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

# For MarkDown columns, assume NaN means no markdown was applied; fill NaNs with 0
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
df[markdown_cols] = df[markdown_cols].fillna(0)

# If there are duplicate holiday columns, drop one (assuming IsHoliday_x is the primary column)
if 'IsHoliday_y' in df.columns:
    df.drop('IsHoliday_y', axis=1, inplace=True)

# Verify cleaning
print(df.isnull().sum())

## 4. Exploratory Data Analysis (EDA)

We perform univariate, bivariate, and multivariate analyses along with computing central tendency, dispersion, skewness, kurtosis, covariance, and correlation.

In [ ]:
# 4.1 Measure of Central Tendency
central_tendency = df.describe().T[['mean', '50%']]
central_tendency.columns = ['Mean', 'Median']

# Calculate mode
mode_values = df.mode().iloc[0]

print("Central Tendency (Mean & Median):\n", central_tendency)
print("\nMode Values:\n", mode_values)

# 4.2 Dispersion of Data
dispersion = df.describe().T[['min', 'max', 'std']]
dispersion['Range'] = dispersion['max'] - dispersion['min']
dispersion['Coefficient of Variation'] = dispersion['std'] / df.mean()

print("\nDispersion of Data:\n", dispersion)

# 4.3 Skewness & Kurtosis
skew_kurt = pd.DataFrame({
    "Skewness": df.skew(),
    "Kurtosis": df.kurt()
})
print("\nSkewness and Kurtosis:\n", skew_kurt)

# 4.4 Covariance & Correlation
print("\nCovariance Matrix:\n", df.cov())
print("\nCorrelation Matrix:\n", df.corr())

# 4.5 Visualization: Univariate Analysis
plt.figure(figsize=(10, 5))
sns.histplot(df['Weekly_Sales'], bins=50, kde=True, color='blue')
plt.title('Distribution of Weekly Sales')
plt.xlabel('Weekly Sales')
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(y=df['Weekly_Sales'])
plt.title('Boxplot of Weekly Sales')
plt.show()

# 4.6 Visualization: Bivariate Analysis
plt.figure(figsize=(8, 5))
sns.scatterplot(x=df['MarkDown1'], y=df['Weekly_Sales'], alpha=0.6)
plt.title('MarkDown1 vs Weekly Sales')
plt.xlabel('MarkDown1')
plt.ylabel('Weekly Sales')
plt.show()

plt.figure(figsize=(8, 6))
sns.boxplot(x='Type', y='Weekly_Sales', data=df)
plt.title('Weekly Sales by Store Type')
plt.show()

# 4.7 Visualization: Multivariate Analysis
sns.pairplot(df[['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']])
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

## 5. Feature Engineering

Create additional features such as date-based components and interaction terms. Also, encode categorical variables if needed.

In [ ]:
# Extract date features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek

# Example: Create an interaction feature between MarkDown1 and IsHoliday
df['MD1_Holiday'] = df['MarkDown1'] * df['IsHoliday_x'].astype(int)

# Encode categorical variable 'Type' using one-hot encoding
df = pd.get_dummies(df, columns=['Type'], drop_first=True)

print(df.head())

## 6. Model Building & Machine Learning

In this section, we build a baseline machine learning model to forecast weekly sales. We will:

- Define the target (`Weekly_Sales`) and features
- Split the data into training and testing sets
- Train a model (using Linear Regression as an example)
- Evaluate the model performance using RMSE, MAE, and R²

In [ ]:
# Define target and features
target = 'Weekly_Sales'
features = df.drop(columns=['Weekly_Sales', 'Date'])

# Split the data (80:20 ratio)
X_train, X_test, y_train, y_test = train_test_split(features, df[target], test_size=0.2, random_state=42)

# Initialize and train the Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = lr_model.predict(X_test)

# Evaluate the model
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R² Score: {r2:.2f}")

## 7. Model Interpretation & Business Insights

- **Feature Importance:** For tree-based models you can extract feature importances or use SHAP values.
- **Residual Analysis:** Visualize residuals to diagnose model fit.
- **Actionable Insights:** Provide recommendations on promotional strategies, store improvements, or other actionable areas based on the model's findings.

In [ ]:
# Plot actual vs predicted Weekly Sales
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.xlabel('Actual Weekly Sales')
plt.ylabel('Predicted Weekly Sales')
plt.title('Actual vs Predicted Weekly Sales')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

# Residual Plot
residuals = y_test - y_pred
plt.figure(figsize=(8,6))
sns.histplot(residuals, kde=True)
plt.title('Residuals Distribution')
plt.xlabel('Residuals')
plt.show()

## 8. Deployment & Reporting

While this project is a proof-of-concept, for deployment you could:

- Package the model as a REST API (using Flask or FastAPI).
- Develop an interactive dashboard (using Plotly Dash or Streamlit).
- Document all findings and create a comprehensive report for stakeholders.

## 9. Conclusion & Next Steps

- **Summary:** We have successfully built a workflow for retail sales forecasting using EDA, feature engineering, and a baseline predictive model.
- **Next Steps:**
  - Experiment with more advanced models (e.g., Random Forest, XGBoost).
  - Fine-tune the model using hyperparameter optimization.
  - Enhance feature engineering based on further domain knowledge.
  - Develop a production-ready system for real-time forecasting.